# ShadowWork-SHIFT | Final Hugging Face publication from Kaggle

**Purpose:** Publish the two **SW12 v3 LoRA adapters** and **SW12–SW16 training/evaluation evidence** into the already-created public Hugging Face repositories, then add the two adapter repos to the existing required **model collection**. This notebook **does not train or evaluate** a model and does not change scientific results.

**Inputs:** Attach the existing Kaggle dataset `shaurya4667/shadowwork-12-closed-loop` (which contains `learner_corrective_v3_adapter/` and `rehearsal_v3_adapter/`, plus training rows/traces). Optionally attach the original SW13, SW14, SW15 and SW16 result ZIPs as a Kaggle input dataset, and/or the curated 5MB `ShadowWork_SHIFT_Publication_Evidence.zip` produced from Project source files. If not attached, SW12 models/data will still publish, but the extended evidence upload will be incomplete. All source checkpoint weights already exist; do **not** rerun experiments.

**Account/secret:** Set a Kaggle Secret named `HF_TOKEN` with a write-enabled Hugging Face access token for `i-am-shaurya05`; enable Internet in Kaggle. Never paste the token into code, notebooks, GitHub or logs. The public Plain SFT base and SW09 adapters have already been verified on HF, so this notebook avoids uploading them again.

**Limit:** Original model and dataset collections already exist. The optional collection-add operations are safe to repeat using `exists_ok=True`.


In [ ]:
%pip install -q "huggingface_hub>=0.34,<2"

## 1. Editable settings and authentication

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
import zipfile
from huggingface_hub import HfApi, login, hf_hub_download
from kaggle_secrets import UserSecretsClient

OWNER = "i-am-shaurya05"
BASE_MODEL_ID = f"{OWNER}/shadowwork-shift-plain-sft"
CORRECTIVE_MODEL_ID = f"{OWNER}/shadowwork-shift-corrective-v3"
REHEARSAL_MODEL_ID = f"{OWNER}/shadowwork-shift-rehearsal-v3"
DATASET_REPO_ID = f"{OWNER}/shadowwork-shift-datasets"

MODEL_COLLECTION_SLUG = f"{OWNER}/shadowwork-shift-models-and-frozen-adapters-6af5c8a5a548fb703ab420ac"
DATA_COLLECTION_SLUG = f"{OWNER}/shadowwork-shift-training-data-and-audit-evidence-6af5c8a51d44c02ee6e19812"

SW12_ROOT = Path("/kaggle/input/datasets/shaurya4667/shadowwork-12-closed-loop")
# Optional directory containing previous result ZIP archives (or nested below it).
RESULTS_ROOT = Path("/kaggle/input")
STAGING_ROOT = Path("/kaggle/working/shift_publication")
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "Add a write-enabled HF_TOKEN in Kaggle Secrets"
api = HfApi(token=HF_TOKEN)
identity = api.whoami()
print("Publishing as:", identity.get("name"))
assert identity.get("name") == OWNER, "HF_TOKEN must belong to the expected account"
assert SW12_ROOT.is_dir(), f"Attach SW12 source dataset: {SW12_ROOT}"

# Verify existing public exact base checkpoint, not a similar upstream model.
base_info = api.model_info(BASE_MODEL_ID, files_metadata=False)
assert not base_info.private and not base_info.gated, "Base checkpoint must be public and ungated"
print("Verified existing public base:", BASE_MODEL_ID)


## 2. Verify original adapter files, publish exact weights with corrected public base linkage

In [ ]:
ADAPTERS = {
    "learner_corrective_v3": (CORRECTIVE_MODEL_ID, "learner_corrective_v3_adapter"),
    "rehearsal_v3": (REHEARSAL_MODEL_ID, "rehearsal_v3_adapter"),
}

for arm, (repo_id, relative_folder) in ADAPTERS.items():
    source_adapter = SW12_ROOT / relative_folder
    assert source_adapter.is_dir(), f"Missing {source_adapter}"
    assert (source_adapter / "adapter_model.safetensors").is_file()
    assert (source_adapter / "adapter_config.json").is_file()
    original_config = json.loads((source_adapter / "adapter_config.json").read_text())
    original_base = original_config.get("base_model_name_or_path", "")
    assert "shadowork_123" in original_base, f"Unexpected base in {arm}: {original_base}"

    model_dir = STAGING_ROOT / "model_releases" / arm
    model_dir.mkdir(parents=True, exist_ok=True)
    for name in ("adapter_model.safetensors", "adapter_config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"):
        file = source_adapter / name
        if file.is_file():
            shutil.copy2(file, model_dir / name)

    original_sha = hashlib.sha256((source_adapter / "adapter_model.safetensors").read_bytes()).hexdigest()
    new_config = dict(original_config)
    new_config["base_model_name_or_path"] = BASE_MODEL_ID
    (model_dir / "adapter_config.json").write_text(json.dumps(new_config, indent=2) + "\n")
    (model_dir / "provenance").mkdir(exist_ok=True)
    (model_dir / "provenance" / "original_adapter_config.json").write_text(json.dumps(original_config, indent=2) + "\n")
    (model_dir / "provenance" / "original_weights_sha256.txt").write_text(original_sha + "\n")

    description = "learner-state corrective training" if arm == "learner_corrective_v3" else "matched ordinary rehearsal training"
    card = f"""---
library_name: peft
pipeline_tag: text-generation
license: cc-by-nc-4.0
base_model: {BASE_MODEL_ID}
base_model_relation: adapter
tags:
- lexsi
- agenttune
- auditkit
- shadowwork-shift
- peft
- lora
---

# ShadowWork-SHIFT: {arm}

This PEFT LoRA adapter is the **SW12 v3 continuation** trained by {description} on synthetic stateful tool-agent workflow tasks. Both v3 arms start from the frozen SW09 corrective adapter and share equal optimization budgets and matched training rows. The exact, non-commercial custom base is [{BASE_MODEL_ID}](https://huggingface.co/{BASE_MODEL_ID}). It cannot be replaced with the upstream Cohere base without changing the scientific comparison.

**Loading:** Use Transformers with the linked custom full checkpoint and PEFT's `PeftModel.from_pretrained(base_model, "{repo_id}")`. The original experiment also needs the task simulator, JSON action format and deterministic controller to reproduce the reported workflow outcomes.

**Training:** 800 rows per arm, 80 optimization steps, LR 5e-5, global batch 8. Source training rows and traces are in the paired [dataset release](https://huggingface.co/datasets/{DATASET_REPO_ID}). Weights are unchanged from the original export. The Kaggle-specific adapter base path was corrected in metadata; its original value and original weights hash are preserved under `provenance/`.

**Evaluation, constant unified_v4 controller over 160 SW12 core/composed task instances:** corrective v3 154/160 strict; rehearsal v3 160/160 strict. With final budgeted_v5 recovery controller: corrective v3 156/160; rehearsal v3 160/160. These are controlled synthetic results; corrective v3 does **not** establish superiority over matched rehearsal or real-world generalization. Original evaluation code: https://github.com/githubshaurya/ShadowWork-SHIFT

**License:** CC BY-NC 4.0 inherited from Cohere Tiny Aya model terms; observe the upstream acceptable use policy. Non-commercial research use only under these terms.
"""
    (model_dir / "README.md").write_text(card)

    api.create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
    print("Uploading", arm, "original weight SHA256", original_sha)
    api.upload_folder(folder_path=str(model_dir), repo_id=repo_id, repo_type="model", commit_message=f"Publish original {arm} adapter and provenance")
    public_info = api.model_info(repo_id)
    public_names = {item.rfilename for item in public_info.siblings}
    assert not public_info.private and not public_info.gated
    assert "adapter_model.safetensors" in public_names and "adapter_config.json" in public_names
    remote_config_path = hf_hub_download(repo_id, "adapter_config.json", repo_type="model", token=HF_TOKEN, force_download=True)
    assert json.loads(Path(remote_config_path).read_text())["base_model_name_or_path"] == BASE_MODEL_ID
    print("Published and verified:", f"https://huggingface.co/{repo_id}")


## 3. Publish SW12 training metadata, datasets, exact scored episodes, and optional SW13–SW16 evidence

In [ ]:
api.create_repo(DATASET_REPO_ID, repo_type="dataset", private=False, exist_ok=True)

records = STAGING_ROOT / "dataset_release"
records.mkdir(exist_ok=True)

# Exact original SW12 rows and scored evaluation JSONs are already mounted as a Kaggle dataset.
# Exclude very large duplicate tokenized Arrow data and all model adapters.
source_files = [
    "all_episode_scorecard.csv", "final_provenance.json", "grouped_summary.csv",
    "precommit.json", "selection_audit.json", "train_tasks.jsonl",
    "dev_tasks.jsonl", "sealed_tasks.jsonl",
    "learner_corrective_v3_training.jsonl", "rehearsal_v3_training.jsonl",
    "learner_corrective_v3_training_metadata.json", "rehearsal_v3_training_metadata.json",
    "learner_corrective_v3_tokenized_source_sha256.txt", "rehearsal_v3_tokenized_source_sha256.txt",
]
for name in source_files:
    file = SW12_ROOT / name
    assert file.is_file(), f"Expected SW12 evidence file: {file}"
    dest = records / "sw12" / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file, dest)

sw12_episodes = sorted((SW12_ROOT / "evaluation").rglob("*.json"))
assert len(sw12_episodes) >= 200, "Could not locate expected SW12 evaluated episodes"
for file in sw12_episodes:
    dest = records / "sw12" / "evaluation" / file.relative_to(SW12_ROOT / "evaluation")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file, dest)
print("SW12 evaluated episodes:", len(sw12_episodes))

# Option A: Attach our curated 4.9-MB evidence ZIP as a Kaggle dataset.
# Option B: Mount individual original SW13–SW16 result ZIPs; the same release will expose their original archives.
curated_archives = sorted(RESULTS_ROOT.rglob("ShadowWork_SHIFT_Publication_Evidence.zip"))
if curated_archives:
    with zipfile.ZipFile(curated_archives[0]) as source_zip:
        assert source_zip.testzip() is None
        source_zip.extractall(records / "archived_evidence")
    print("Included curated complete SW09/SW13–SW16 evidence:", curated_archives[0])
else:
    names = (
        "ShadowWork_13_Live_Results.zip",
        "ShadowWork_14_Controller_Repair_Results1.zip",
        "ShadowWork_15_Cross_Protocol_Recovery_Results.zip",
        "ShadowWork_16_Unified_Recovery_Results.zip",
        "ShadowWork_16_Budget_Regression_Results.zip",
    )
    missing = []
    for name in names:
        matches = sorted(RESULTS_ROOT.rglob(name))
        if not matches:
            missing.append(name)
            continue
        dest = records / "archived_evidence" / "source_archives" / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(matches[0], dest)
    if missing:
        print("WARNING: optional evidence archives absent:", missing)

readme = f"""---
license: cc-by-nc-4.0
pretty_name: ShadowWork-SHIFT SW12–SW16 Self-Healing Training and Evaluations
tags:
- agents
- agenttune
- auditkit
- lexsi
- safety
---

# ShadowWork-SHIFT | SW12–SW16 training and recovery evidence

Exact synthetic workflow training rows, the SW12 task splits and scored evaluation episodes, and archived SW09/SW13–SW16 trace evidence where attached. This is supplementary to the already-published [SW09 source training and holdout dataset](https://huggingface.co/datasets/{OWNER}/shadowwork-shift-evidence-and-training).

**Method:** v3 corrective vs compute/row-matched rehearsal, 800 rows per arm, 80 optimizer steps, same original SW09 corrective LoRA start; evaluate with fixed controllers and deterministic state/evidence scoring. SW12 sealed: original frozen 24/64, corrective v3 46/64, rehearsal v3 47/64 strict. SW16 unified_v4: 132/160, 154/160, 160/160. SW16 budget_v5: 156/160 corrective v3 and 160/160 rehearsal v3. The final regression repair is **not** an untouched external holdout, and rehearsal is competitive or superior throughout.

**Data:** Synthetic configuration-maintenance workflows and failure injections, not real consumer data. Includes tool traces and stored-state outcomes. Source experiment notebooks and full limitations: https://github.com/githubshaurya/ShadowWork-SHIFT

**License:** CC BY-NC 4.0 chosen for the research bundle and compatible with inherited non-commercial base-model constraints. Ensure upstream and third-party terms are respected.
"""
(records / "README.md").write_text(readme)

api.upload_folder(folder_path=str(records), repo_id=DATASET_REPO_ID, repo_type="dataset", commit_message="Publish verified SW12–SW16 training and evaluation evidence")
public_dataset = api.dataset_info(DATASET_REPO_ID)
public_files = {item.rfilename for item in public_dataset.siblings}
assert not public_dataset.private and "sw12/train_tasks.jsonl" in public_files
assert "sw12/grouped_summary.csv" in public_files
print("Dataset publication verified:", f"https://huggingface.co/datasets/{DATASET_REPO_ID}")


## 4. Add published repositories to the required existing public collections

In [ ]:
for repo_id in (CORRECTIVE_MODEL_ID, REHEARSAL_MODEL_ID):
    api.add_collection_item(collection_slug=MODEL_COLLECTION_SLUG, item_id=repo_id, item_type="model", exists_ok=True)

api.add_collection_item(collection_slug=DATA_COLLECTION_SLUG, item_id=DATASET_REPO_ID, item_type="dataset", exists_ok=True)

for slug in (MODEL_COLLECTION_SLUG, DATA_COLLECTION_SLUG):
    collection = api.get_collection(slug)
    print("Collection verified:", f"https://huggingface.co/collections/{slug}")
    print("Items:", [(item.item_type, item.item_id) for item in collection.items])

print("SUCCESS: Models/datasets published, verified, and added to collections.")
print("GitHub: https://github.com/githubshaurya/ShadowWork-SHIFT")
